In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, confusion_matrix


In [2]:
df = pd.read_csv("../data/processed/v3/development_v3.csv")

# FIX OBBLIGATORIO
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("unknown").astype(str)

X = df.drop(columns=["label"])
y = df["label"]


In [ ]:
preprocess = ColumnTransformer(
	transformers=[
		("article_tfidf", TfidfVectorizer(
			max_features=50_000,
			ngram_range=(1, 2),
			min_df=3,
			max_df=0.9,
			stop_words="english",
			sublinear_tf=True
		), "article"),

		("title_tfidf", TfidfVectorizer(
			max_features=20_000,
			ngram_range=(1, 3),
			min_df=2,
			max_df=0.9,
			stop_words="english",
			sublinear_tf=True
		), "title"),

		("source", OneHotEncoder(handle_unknown="ignore"), ["source"])
	],
	n_jobs=-1
)


In [4]:
model = Pipeline([
	("prep", preprocess),
	("clf", LogisticRegression(
		C=1.0,
		max_iter=1000,
		n_jobs=-1
	))
])


In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1s = []
recalls = []
cms = []

for tr, te in skf.split(X, y):
	model.fit(X.iloc[tr], y.iloc[tr])
	yp = model.predict(X.iloc[te])

	f1s.append(f1_score(y.iloc[te], yp, average="macro"))
	recalls.append(recall_score(y.iloc[te], yp, average="macro"))
	cms.append(confusion_matrix(y.iloc[te], yp))

print("SEMANTIC-ONLY (article + title + source)")
print("Macro F1:", np.mean(f1s))
print("Macro Recall:", np.mean(recalls))
print("Confusion Matrix:\n", np.sum(cms, axis=0))


SEMANTIC-ONLY (article + title + source)
Macro F1: 0.701586734528753
Macro Recall: 0.697717961690693
Confusion Matrix:
 [[18887   633   406   781   195  2410   229]
 [  757  8306   536   352    69   455   113]
 [  668   625  9096   344    45   273   110]
 [ 1642   559   519  5007   610  1429   211]
 [  241    57    17   330  7574   351     4]
 [ 3760   625   298  1148   627  6329   266]
 [  434   146    83   216    36   288  1899]]
